In [0]:
%run "../common functions"

In [0]:
from pyspark.sql.functions import *

In [0]:
jobs_df = spark.table("raw_catalogue.jobs.workday")

In [0]:
todays_jobs = jobs_df.filter(col("days") == 0).dropDuplicates(["url"])
todays_jobs = todays_jobs.withColumn("hrs_ago", lit("Today")) \
                        .withColumnRenamed("locationsText", "location")
todays_jobs = todays_jobs.select("title", "url", "hrs_ago", "location", "company", "api_url")
display(todays_jobs)

In [0]:
scored_jobs = score_job_workday(todays_jobs, "url")
scored_df = spark.createDataFrame(scored_jobs)
scored_jobs_df = scored_df.join(todays_jobs, on=['url'])
filtered_df = scored_jobs_df.filter(scored_jobs_df.score > 70).dropDuplicates(['url'])
display(filtered_df)

In [0]:
filtered_df.write.mode('overwrite').option("mergeSchema", "true").saveAsTable(f"staging_catalogue.jobs.workday")

In [0]:
new_jobs = spark.sql("select a.* from staging_catalogue.jobs.workday a left anti join main_catalogue.jobs.workday b on a.url = b.url")
new_jobs = new_jobs.orderBy(col("score").desc())

In [0]:
users_df = spark.table('main_catalogue.jobs.users')
chat_ids = users_df.filter(users_df.skill == "Data Engineer").select(collect_list('chat_id')).collect()[0][0]

In [0]:
form_message_and_send(new_jobs, chat_ids)

In [0]:
new_jobs.createOrReplaceTempView("new_jobs")

In [0]:
spark.sql("insert into main_catalogue.jobs.workday select * from new_jobs")